# MaCAD S.3 — Assignment 1
## Building Graph Generation: Double-L Residential Building with Pilotis

**Course workflow:** BGR — Building Graph Representation (`S06-13A GML Creating BGR Graph`)  
**Project:** Double-L residential massing with ground-level pilotis  
**Architectural question:** How do the topological relationships between structural pilotis,
residential massing, and vertical cores reveal the building's spatial organisation?

---

**OBJ inputs — export these four files from Rhino before running:**

| File | Rhino geometry | Cell-type code |
|------|---------------|----------------|
| `ground.obj` | Ground slab Brep(s) | 0 |
| `columns.obj` | Pilotis column Breps | 1 |
| `offices.obj` | Residential massing Breps (course-standard filename) | 3 |
| `core.obj` | Vertical core Breps, one per floor | 4 |

> **Note:** `offices.obj` follows the BGR course naming convention. In this project it contains **residential massing**, not office space.  
> Category 2 (plinth) is defined in the feature schema but has **zero nodes** in this building — stated explicitly rather than invented.

### Assignment 1 Requirements Checklist

- [ ] Four closed-solid OBJ files exported from Rhino (ground, columns, offices, core)
- [ ] OBJs imported with `Topology.ByOBJPath`
- [ ] Every cell assigned `cell_type`, `cell_name`, `cell_color` dictionaries
- [ ] All cells merged into a single `CellComplex` topology model
- [ ] Dictionaries transferred back onto merged cells via `TransferDictionariesBySelectors`
- [ ] Adjacency graph built with `Graph.ByTopology`
- [ ] `cell_type` one-hot encoded into `feature_00` ... `feature_04` on every graph vertex
- [ ] Graph dataset exported to CSV (`graphs.csv`, `nodes.csv`, `edges.csv`)
- [ ] Verification summary printed (node count, edge count, counts by type, isolated nodes)
- [ ] Graph visualisation saved to `05_visuals/`

---
## Setup

In [ ]:
# Uncomment and run once if topologicpy is not installed:
# !pip install topologicpy

from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.Color import Color
from topologicpy.Helper import Helper

import os
import warnings
warnings.filterwarnings("ignore")

print("topologicpy imported successfully")

In [ ]:
# ── Path configuration ─────────────────────────────────────────────────────
# Set PROJECT_ROOT to the assignment_01_graph_generation folder on your machine.
# All other paths are derived from it.

PROJECT_ROOT = "D:/GitHub/GML_Edu/assignment_01_graph_generation"

ASSETS_DIR  = os.path.join(PROJECT_ROOT, "assets")
OUTPUTS_DIR = os.path.join(PROJECT_ROOT, "outputs")
VISUALS_DIR = os.path.join(PROJECT_ROOT, "05_visuals")

os.makedirs(VISUALS_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

for label, path in [("Assets ", ASSETS_DIR), ("Outputs", OUTPUTS_DIR), ("Visuals", VISUALS_DIR)]:
    status = "OK" if os.path.isdir(path) else "MISSING — create folder and add OBJ files before running"
    print(f"{label}: {path}  [{status}]")

---
## Step 1 — Import OBJ Files

Each OBJ file corresponds to one architectural layer.  
`Topology.ByOBJPath` returns a list of topology objects — one per named group in the OBJ file.  
`transposeAxes=False` keeps the Rhino coordinate system; the OBJ exporter has already applied the
Rhino Z-up → OBJ Y-up axis remap.

In [ ]:
ground_objs = Topology.ByOBJPath(os.path.join(ASSETS_DIR, "ground.obj"),  transposeAxes=False)
column_objs = Topology.ByOBJPath(os.path.join(ASSETS_DIR, "columns.obj"), transposeAxes=False)
office_objs = Topology.ByOBJPath(os.path.join(ASSETS_DIR, "offices.obj"), transposeAxes=False)
core_objs   = Topology.ByOBJPath(os.path.join(ASSETS_DIR, "core.obj"),    transposeAxes=False)

print(f"ground.obj  : {len(ground_objs)} object(s)")
print(f"columns.obj : {len(column_objs)} object(s)")
print(f"offices.obj : {len(office_objs)} object(s)  [residential massing]")
print(f"core.obj    : {len(core_objs)} object(s)   [per-floor core volumes]")

In [ ]:
# Visualise all imported objects together — confirm geometry is present and correctly positioned.
Topology.Show(ground_objs, column_objs, office_objs, core_objs)

---
## Step 2 — Assign Cell-Type Categories

Each architectural layer is converted to a set of **cells** (closed 3-D volumes).  
For every cell we create a **selector**: an `InternalVertex` (centroid) carrying a `Dictionary`
with three keys:

| Key | Value |
|-----|-------|
| `cell_type` | Integer code (0–4) |
| `cell_name` | Human-readable label |
| `cell_color` | Display colour string |

**Category codes for this building:**

| Code | Name | Colour | Note |
|------|------|--------|------|
| 0 | ground | green | Ground slab |
| 1 | columns | lightgray | Pilotis |
| 2 | plinth | gold | Not present — 0 nodes |
| 3 | residential | steelblue | Massing (file: offices.obj) |
| 4 | core | firebrick | Per-floor core volumes |

In [ ]:
# Initialise the selectors list — populated by each layer below.
selectors = []

# ── Ground (cell_type = 0) ────────────────────────────────────────────────
ground_faces = [Topology.Faces(obj) for obj in ground_objs
                if Topology.IsInstance(obj, "Topology")]
ground_faces = Helper.Flatten(ground_faces)
ground       = Topology.SelfMerge(Cluster.ByTopologies(ground_faces))
ground_cells = Topology.Cells(ground)

for cell in ground_cells:
    d = Dictionary.ByKeysValues(
        ["cell_type", "cell_name", "cell_color"],
        [0, "ground", "green"]
    )
    s = Topology.InternalVertex(cell)
    s = Topology.SetDictionary(s, d)
    selectors.append(s)

print(f"Ground cells   : {len(ground_cells)}")

In [ ]:
# ── Columns / pilotis (cell_type = 1) ─────────────────────────────────────
column_faces = [Topology.Faces(obj) for obj in column_objs
                if Topology.IsInstance(obj, "Topology")]
column_faces = Helper.Flatten(column_faces)
columns      = Topology.SelfMerge(Cluster.ByTopologies(column_faces))
column_cells = Topology.Cells(columns)

for cell in column_cells:
    d = Dictionary.ByKeysValues(
        ["cell_type", "cell_name", "cell_color"],
        [1, "columns", "lightgray"]
    )
    s = Topology.InternalVertex(cell)
    s = Topology.SetDictionary(s, d)
    selectors.append(s)

print(f"Column cells   : {len(column_cells)}")

In [ ]:
# ── Residential massing (cell_type = 3) — file: offices.obj ───────────────
# 'offices.obj' is the BGR course-standard filename.
# In this project it contains residential massing volumes, not offices.
office_faces = [Topology.Faces(obj) for obj in office_objs
                if Topology.IsInstance(obj, "Topology")]
office_faces = Helper.Flatten(office_faces)
offices      = Topology.SelfMerge(Cluster.ByTopologies(office_faces))
office_cells = Topology.Cells(offices)

for cell in office_cells:
    d = Dictionary.ByKeysValues(
        ["cell_type", "cell_name", "cell_color"],
        [3, "residential", "steelblue"]
    )
    s = Topology.InternalVertex(cell)
    s = Topology.SetDictionary(s, d)
    selectors.append(s)

print(f"Residential cells : {len(office_cells)}  [from offices.obj]")

In [ ]:
# ── Core volumes (cell_type = 4) — one per floor ──────────────────────────
# Core geometry must be per-floor closed solids, not a single full-height Brep.
core_faces = [Topology.Faces(obj) for obj in core_objs
              if Topology.IsInstance(obj, "Topology")]
core_faces = Helper.Flatten(core_faces)
cores      = Topology.SelfMerge(Cluster.ByTopologies(core_faces))
core_cells = Topology.Cells(cores)

for cell in core_cells:
    d = Dictionary.ByKeysValues(
        ["cell_type", "cell_name", "cell_color"],
        [4, "core", "firebrick"]
    )
    s = Topology.InternalVertex(cell)
    s = Topology.SetDictionary(s, d)
    selectors.append(s)

print(f"Core cells     : {len(core_cells)}  [per-floor volumes]")

In [ ]:
# Summary of selectors before merging.
# Category 2 (plinth) is intentionally absent from this building.
print("Selectors created:")
print(f"  type 0  ground       : {len(ground_cells)}")
print(f"  type 1  columns      : {len(column_cells)}")
print(f"  type 2  plinth       : 0  [not present in Double-L design]")
print(f"  type 3  residential  : {len(office_cells)}")
print(f"  type 4  core         : {len(core_cells)}")
print(f"  Total selectors      : {len(selectors)}")

---
## Step 3 — CellComplex (Merge All Volumes)

`Topology.SelfMerge` on a `Cluster` of all cells produces a single `CellComplex` — the unified
topology model from which the graph will be derived.  
Adjacent cells share faces; the merge resolves shared boundaries into a coherent volumetric model.

In [ ]:
all_cells = ground_cells + column_cells + office_cells + core_cells
model = Topology.SelfMerge(Cluster.ByTopologies(all_cells))

model_cells = Topology.Cells(model)
print(f"CellComplex type : {type(model).__name__}")
print(f"Total cells      : {len(model_cells)}")

# Visualise the merged topology (no colours yet — dictionaries not yet transferred)
Topology.Show(model_cells)

---
## Step 4 — Transfer Dictionaries

`Topology.TransferDictionariesBySelectors` uses each selector vertex to locate the cell in the
merged model that contains it, then copies the dictionary onto that cell.  
After transfer, every model cell has its `cell_type`, `cell_name`, and `cell_color`.

In [ ]:
model = Topology.TransferDictionariesBySelectors(model, selectors, tranCells=True)

# Verify: sample the first cell
sample_cells = Topology.Cells(model)
if sample_cells:
    d_sample = Topology.Dictionary(sample_cells[0])
    print("Sample cell dictionary:")
    print(f"  keys   : {Dictionary.Keys(d_sample)}")
    print(f"  values : {Dictionary.Values(d_sample)}")

# Visualise with category colours — confirms successful dictionary transfer
Topology.Show(Topology.Cells(model), faceColorKey="cell_color", faceOpacity=0.85)

---
## Step 5 — Build Adjacency Graph

`Graph.ByTopology` derives a graph from the `CellComplex`: one vertex per cell, one edge per
pair of cells that share a face (topological adjacency).

Each vertex then receives **one-hot encoded** feature columns (`feature_00` ... `feature_04`)
derived from `cell_type`. These columns are required by the BGR training pipeline.

In [ ]:
def one_hot_encode(value, n):
    """Return a one-hot list of length n for the given integer category value."""
    if value is None or not isinstance(value, (int, float)):
        return [0] * n
    value = int(value)
    if value < 0 or value >= n:
        return [0] * n
    vector = [0] * n
    vector[value] = 1
    return vector


# Feature column names — one per category (0–4)
feature_names = ["feature_" + str(i).zfill(2) for i in range(5)]
print("Feature names:", feature_names)
print()
print("Category mapping:")
print("  feature_00  ground       (type 0)")
print("  feature_01  columns      (type 1)")
print("  feature_02  plinth       (type 2) — zero nodes in this building")
print("  feature_03  residential  (type 3) — offices.obj")
print("  feature_04  core         (type 4)")

In [ ]:
# Build the graph from the merged CellComplex
graph    = Graph.ByTopology(model)
vertices = Graph.Vertices(graph)
edges_g  = Graph.Edges(graph)

print(f"Graph vertices (nodes) : {len(vertices)}")
print(f"Graph edges            : {len(edges_g)}")

# Assign one-hot feature columns to every vertex
for v in vertices:
    d         = Topology.Dictionary(v)
    cell_type = Dictionary.ValueAtKey(d, "cell_type")
    ohe       = one_hot_encode(cell_type, 5)
    for i, fname in enumerate(feature_names):
        d = Dictionary.SetValueAtKey(d, fname, ohe[i])
    v = Topology.SetDictionary(v, d)

print("One-hot features assigned to all vertices.")

In [ ]:
# Verify features on the first few vertices
print("Vertex feature verification (first 5 nodes):")
print(f"{'node':>4}  {'cell_type':>9}  {'cell_name':>12}  {'one-hot features'}")
print("-" * 60)

for idx, v in enumerate(vertices[:5]):
    d  = Topology.Dictionary(v)
    ct = Dictionary.ValueAtKey(d, "cell_type")
    cn = Dictionary.ValueAtKey(d, "cell_name")
    ohe_check = [Dictionary.ValueAtKey(d, fname) for fname in feature_names]
    print(f"{idx:>4}  {str(ct):>9}  {str(cn):>12}  {ohe_check}")

---
## Step 6 — Export Graph Dataset (CSV)

`Graph.ExportToCSV` writes three CSV files to `OUTPUTS_DIR`:

| File | Contents |
|------|----------|
| `graphs.csv` | One row per graph — `graph_id`, `label`, `feat` |
| `nodes.csv` | One row per node — `graph_id`, `node_id`, `label`, feature columns, `X`, `Y`, `Z` |
| `edges.csv` | One row per edge — `graph_id`, `src_id`, `dst_id`, `label` |

In [ ]:
status = Graph.ExportToCSV(
    graph,
    path=OUTPUTS_DIR,
    nodeFeaturesKeys=feature_names,
    overwrite=True
)

print(f"Export status : {status}")
print(f"Output folder : {OUTPUTS_DIR}")

for fname in ["graphs.csv", "nodes.csv", "edges.csv"]:
    fpath = os.path.join(OUTPUTS_DIR, fname)
    exists = "written" if os.path.isfile(fpath) else "NOT FOUND"
    print(f"  {fname:12s}: {exists}")

---
## Step 7 — Verification Summary

Load the exported CSVs and use NetworkX to compute connectivity statistics.

In [ ]:
import pandas as pd
import networkx as nx

nodes_df = pd.read_csv(os.path.join(OUTPUTS_DIR, "nodes.csv"))
edges_df = pd.read_csv(os.path.join(OUTPUTS_DIR, "edges.csv"))
graphs_df = pd.read_csv(os.path.join(OUTPUTS_DIR, "graphs.csv"))

# Filter to graph_id = 0 (single building)
g0_nodes = nodes_df[nodes_df["graph_id"] == 0].reset_index(drop=True)
g0_edges = edges_df[edges_df["graph_id"] == 0].reset_index(drop=True)

# Build NetworkX graph for analysis
G = nx.Graph()
for _, row in g0_nodes.iterrows():
    G.add_node(int(row["node_id"]),
               cell_type=int(row["label"]),
               X=float(row["X"]), Y=float(row["Y"]), Z=float(row["Z"]))
for _, row in g0_edges.iterrows():
    G.add_edge(int(row["src_id"]), int(row["dst_id"]))

# Counts by cell_type
type_names  = {0: "ground", 1: "columns", 2: "plinth", 3: "residential", 4: "core"}
type_counts = g0_nodes.groupby("label").size().to_dict()

n_components = nx.number_connected_components(G)
n_isolated   = len(list(nx.isolates(G)))

print("=" * 52)
print("GRAPH VERIFICATION SUMMARY")
print("=" * 52)
print(f"Topology cells (model)  : {len(Topology.Cells(model))}")
print(f"Graph nodes             : {G.number_of_nodes()}")
print(f"Graph edges             : {G.number_of_edges()}")
print("-" * 52)
print("Nodes by cell_type:")
for k in range(5):
    count = type_counts.get(k, 0)
    note = "  [no plinth in this building]" if k == 2 and count == 0 else ""
    print(f"  {k}  {type_names[k]:14s}: {count}{note}")
print("-" * 52)
print(f"Connected components    : {n_components}")
print(f"Isolated nodes          : {n_isolated}")
print("=" * 52)

---
## Step 8 — Graph Visualisation

Two views are produced:
1. **Coloured massing** — `Topology.Show` on the merged model cells, coloured by `cell_color`.
2. **Adjacency graph** — `Topology.Show` on the graph object; vertices positioned at cell centroids,
   edges represent topological adjacency.

Figures are saved to `05_visuals/` as PNG (requires `kaleido`: `pip install kaleido`).  
If kaleido is not installed, use the camera icon in the plotly toolbar to export manually.

In [ ]:
# ── Coloured massing ──────────────────────────────────────────────────────
massing_fig = Topology.Show(
    Topology.Cells(model),
    faceColorKey="cell_color",
    faceOpacity=0.85,
    renderer=None
)

if massing_fig is not None:
    massing_fig.show()
    try:
        massing_fig.write_image(
            os.path.join(VISUALS_DIR, "01_massing_colored.png"),
            width=1400, height=1000
        )
        print("Saved: 01_massing_colored.png")
    except Exception as e:
        print(f"PNG save skipped — install kaleido: pip install kaleido")
        print(f"Error: {e}")
else:
    # Fallback: direct render
    Topology.Show(Topology.Cells(model), faceColorKey="cell_color", faceOpacity=0.85)

In [ ]:
# ── Adjacency graph ───────────────────────────────────────────────────────
graph_fig = Topology.Show(graph, renderer=None)

if graph_fig is not None:
    graph_fig.show()
    try:
        graph_fig.write_image(
            os.path.join(VISUALS_DIR, "02_graph_topologic.png"),
            width=1400, height=1000
        )
        print("Saved: 02_graph_topologic.png")
    except Exception as e:
        print(f"PNG save skipped — install kaleido: pip install kaleido")
        print(f"Error: {e}")
else:
    Topology.Show(graph)

---
## Next Step

Open `A1_02_DoubleL_Graph_Visualisation_and_Export.ipynb` to:
- Load the exported CSVs
- Produce annotated 2-D plan and 3-D graph figures
- Review the final assignment checklist
- Generate a report-ready markdown summary

The CSV files at `outputs/` are also the input for the BGR prediction pipeline (`S06-13B`).